In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, average_precision_score
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.inspection import PartialDependenceDisplay
from xgboost import XGBClassifier
from dcurves import dca, plot_graphs

MOVER = r'C:\Users\shahb\Desktop\Research\Datasets\MOVER'
df = pd.read_csv(MOVER + r'\analytic_full.csv')
df['SEX_encoded'] = (df['SEX'] == 'Male').astype(int)

feats_m2 = ['AGE','SEX_encoded','ASA_RATING_C','OPERATIVE_DURATION',
            'hypertension','diabetes','chf','cad','copd','ckd',
            'prior_mi','afib','stroke','pvd',
            'HR_mean','HR_min','HR_max','HR_std','SpO2_mean','SpO2_min']

y = df['CARDIAC_OUTCOME']
X2 = df[feats_m2].copy()
X2['ASA_RATING_C'] = X2['ASA_RATING_C'].fillna(X2['ASA_RATING_C'].median())

X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y, test_size=.2,
                                                random_state=42, stratify=y)
spw = len(y2_tr)/y2_tr.sum()
xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=.05,
                    scale_pos_weight=spw, random_state=42, eval_metric='auc')
xgb.fit(X2_tr, y2_tr)
xgb_proba = xgb.predict_proba(X2_te)[:,1]
print('XGB AUC:', roc_auc_score(y2_te, xgb_proba).round(3))

In [ ]:
# temporal validation - train on pre-covid, test on covid era
df['SURGERY_DATE'] = pd.to_datetime(df['SURGERY_DATE'], infer_datetime_format=True)
tr_t = df[df['SURGERY_DATE'].dt.year <= 2020]
te_t = df[df['SURGERY_DATE'].dt.year >= 2021]

Xtr_t = tr_t[feats_m2].copy()
Xtr_t['ASA_RATING_C'] = Xtr_t['ASA_RATING_C'].fillna(Xtr_t['ASA_RATING_C'].median())
yte_t = te_t['CARDIAC_OUTCOME']
Xte_t = te_t[feats_m2].copy()
Xte_t['ASA_RATING_C'] = Xte_t['ASA_RATING_C'].fillna(Xtr_t['ASA_RATING_C'].median())

xgb_t = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=.05,
                       scale_pos_weight=len(tr_t)/tr_t['CARDIAC_OUTCOME'].sum(),
                       random_state=42, eval_metric='auc')
xgb_t.fit(Xtr_t, tr_t['CARDIAC_OUTCOME'])
proba_t = xgb_t.predict_proba(Xte_t)[:,1]
print(f'temporal AUC: {roc_auc_score(yte_t, proba_t):.3f}')
print(f'train: {len(tr_t)} cases, {tr_t["CARDIAC_OUTCOME"].sum()} events')
print(f'test: {len(te_t)} cases, {yte_t.sum()} events')

In [ ]:
# platt scaling calibration
cal = CalibratedClassifierCV(xgb, method='sigmoid', cv='prefit')
cal.fit(X2_tr, y2_tr)
cal_proba = cal.predict_proba(X2_te)[:,1]

In [ ]:
# dca - manual implementation, dcurves library was buggy
def net_benefit(y_true, y_pred, thresh):
    n = len(y_true)
    tp = ((y_pred >= thresh) & (y_true == 1)).sum()
    fp = ((y_pred >= thresh) & (y_true == 0)).sum()
    return (tp/n) - (fp/n) * (thresh/(1-thresh))

def nb_all(y_true, thresh):
    n = len(y_true)
    return (y_true.sum()/n) - ((y_true==0).sum()/n) * (thresh/(1-thresh))

thresholds = np.arange(0.005, 0.20, 0.005)
nb_lr_vals  = [net_benefit(y2_te.values, xgb_proba, t) for t in thresholds]
nb_cal_vals = [net_benefit(y2_te.values, cal_proba, t) for t in thresholds]
nb_all_vals = [nb_all(y2_te, t) for t in thresholds]

fig, ax = plt.subplots(figsize=(9,6))
ax.plot(thresholds, [0]*len(thresholds), 'k--', lw=2, label='Treat none')
ax.plot(thresholds, nb_all_vals, 'r-', lw=2, label='Treat all')
ax.plot(thresholds, nb_lr_vals, 'b-', lw=2, label='Model 1 LR preop only')
ax.plot(thresholds, nb_cal_vals, 'g-', lw=2.5, label='Model 2 XGB calibrated')
ax.set(xlabel='Threshold Probability', ylabel='Net Benefit',
       title='Decision Curve Analysis', xlim=(0.005,.20), ylim=(-.005,.006))
ax.legend(fontsize=11)
ax.grid(True, alpha=.3)
plt.tight_layout()
plt.savefig(MOVER + r'\figures\figure7_dca.png', dpi=300)
plt.show()

In [ ]:
# partial dependence plot - HR_mean u shape
fig, ax = plt.subplots(figsize=(8,5))
PartialDependenceDisplay.from_estimator(xgb, X2_te, ['HR_mean'], ax=ax)
ax.set_title('PDP — mean intraoperative HR', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean HR (bpm)', fontsize=12)
ax.set_ylabel('Partial dependence', fontsize=12)
ax.grid(True, alpha=.3)
plt.tight_layout()
plt.savefig(MOVER + r'\figures\figure_s1_pdp_hr.png', dpi=300)
plt.show()

In [ ]:
# sensitivity specificity at thresholds
for thresh, label in [(.02, '2%'), (.05, '5%')]:
    ybin = (cal_proba >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y2_te, ybin).ravel()
    print(f'\n{label} threshold:')
    print(f'  sens={tp/(tp+fn):.3f} spec={tn/(tn+fp):.3f}')
    print(f'  PPV={tp/(tp+fp) if tp+fp>0 else 0:.3f} NPV={tn/(tn+fn):.3f}')

In [ ]:
# auc-pr
print('AUC-PR LR:', average_precision_score(y2_te, xgb_proba).round(3))
print('AUC-PR XGB cal:', average_precision_score(y2_te, cal_proba).round(3))
# random classifier baseline at 0.77% event rate = 0.0077

In [ ]:
# bootstrap CIs
def boot_auc(y_true, y_pred, n=1000):
    aucs = []
    for _ in range(n):
        idx = np.random.randint(0, len(y_true), len(y_true))
        if y_true.iloc[idx].sum() == 0: continue
        aucs.append(roc_auc_score(y_true.iloc[idx], y_pred[idx]))
    return np.percentile(aucs, [2.5, 97.5])

ci = boot_auc(y2_te, xgb_proba)
print(f'XGB AUC: {roc_auc_score(y2_te, xgb_proba):.3f} ({ci[0]:.3f}-{ci[1]:.3f})')

In [ ]:
# subgroup analysis by surgery type
cats = {
    'General Surgery': ['laparoscop','laparotomy','cholecystectomy','appendectomy','colectomy','hernia'],
    'Orthopedic': ['arthroplasty','orif','arthroscop','fracture','spine'],
    'Vascular': ['fistula','fistulogram','angioplasty','vascular','aortic'],
    'Oncologic': ['resection','mastectomy','tumor','excision'],
    'Urologic': ['prostatectomy','cystoscopy','nephrectomy'],
    'Gynecologic': ['hysterectomy','myomectomy','ovarian']
}

df['cat'] = 'Other'
for cat, kws in cats.items():
    m = df['PRIMARY_PROCEDURE_NM'].str.lower().str.contains('|'.join(kws), na=False)
    df.loc[m, 'cat'] = cat

# evaluate on test set only
te_idx = X2_te.index
for cat in ['Other', 'General Surgery', 'Vascular', 'Orthopedic', 'Urologic', 'Oncologic', 'Gynecologic']:
    sub = df.loc[te_idx][df.loc[te_idx, 'cat'] == cat]
    if sub['CARDIAC_OUTCOME'].sum() < 10:
        print(f'{cat}: {len(sub)} cases, {sub["CARDIAC_OUTCOME"].sum()} events - insufficient')
        continue
    Xsub = X2_te.loc[sub.index]
    ysub = sub['CARDIAC_OUTCOME']
    p = xgb.predict_proba(Xsub)[:,1]
    print(f'{cat}: n={len(sub)}, events={ysub.sum()}, AUC={roc_auc_score(ysub,p):.3f}')

In [ ]:
# complete lab sensitivity analysis for model 1
from sklearn.linear_model import LogisticRegression

lab_cols = ['Creatinine','Hemoglobin','Platelets','Sodium','Potassium','Hematocrit']
complete = df.dropna(subset=lab_cols).copy()
print(f'complete lab cases: {len(complete)}, events: {complete["CARDIAC_OUTCOME"].sum()}')

feats_preop = ['AGE','SEX_encoded','ASA_RATING_C','OPERATIVE_DURATION',
               'hypertension','diabetes','chf','cad','copd',
               'ckd','prior_mi','afib','stroke','pvd']
Xc = complete[feats_preop].copy()
Xc['ASA_RATING_C'] = Xc['ASA_RATING_C'].fillna(Xc['ASA_RATING_C'].median())
yc = complete['CARDIAC_OUTCOME']

Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=.2,
                                                 random_state=42, stratify=yc)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(Xc_tr, yc_tr)
lr_p = lr.predict_proba(Xc_te)[:,1]
ci_c = boot_auc(yc_te, lr_p)
print(f'model 1 complete labs AUC: {roc_auc_score(yc_te,lr_p):.3f} ({ci_c[0]:.3f}-{ci_c[1]:.3f})')